# Ontario ignition EDA
Full-file aggregates plus a uniform sampled analysis. No modelling, imputation or row deletion. Run `scripts/run_ignition_eda.py` to regenerate cached tables; scanning the CSV reads 6.54 GB. The HTML report includes all figures and explanations.

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
# Works when Jupyter starts in this folder or any parent inside the checkout.
EDA = next((p for root in [Path.cwd(), *Path.cwd().parents] for p in [root, root / "docs/eda"] if (p / "eda_summary.json").is_file()), None)
if EDA is None:
    raise FileNotFoundError("Start Jupyter inside the COMP-385 checkout.")
summary = json.loads((EDA / "eda_summary.json").read_text(encoding="utf-8"))
display({k: summary[k] for k in ["rows","positive_cell_days","ignition_prevalence_pct","grid_cells","dates"]})

In [ ]:
display(pd.read_csv(EDA / "annual_summary.csv"))
display(pd.read_csv(EDA / "monthly_summary.csv"))

In [ ]:
display(pd.read_csv(EDA / "missingness.csv").sort_values("missing_pct", ascending=False))

In [ ]:
display(pd.read_csv(EDA / "feature_dictionary.csv"))

In [ ]:
display(pd.read_csv(EDA / "feature_statistics.csv"))
display(pd.read_csv(EDA / "strong_correlations.csv"))

In [ ]:
for chart in ["eda_overview.png","feature_missingness.png","spatial_coverage.png","class_distributions.png","feature_correlations.png"]:
    display(Image(filename=str(EDA / chart)))

This panel estimates whether at least one recorded ignition occurs in a 10 km cell on a target local calendar day. Issue time is 00:00 America/Toronto. Weather is from t-1; seven-day rain uses seven completed preceding dates; ignition histories exclude the target date. Archives lack historical per-record available_at times. This is retrospective hindcast, not a validated operational forecast replay.

Logistic regression: training-only imputation, numeric scaling and explicit missingness/censoring indicators. Histogram gradient boosting: preserve NaNs and quality fields. The baseline uses training-label prevalence and no predictors. The 36 columns are source inputs; derived indicators may increase the final matrix width.

HRDPS, ERA5-Land, fuel type, vegetation density, lightning, roads and settlement are not present in this v1 matrix. The PDF describes proposed future work. For HRDPS, match variables, units, grid, forecast lead times and issue timing explicitly; a same-day hour 0-6 live forecast cannot silently replace prior-day observations. CWFIS-derived FWI and seven-day rain also require a documented calculation/initialisation contract. Wind direction and a dated fuel layer are required separately for spread work. Existing fires should feed a separate incident-conditioned spread simulator.

The listed full-file checks cover schema parsing, keys/universe, labels/count consistency, timing strings, selected bounds and missingness relationships. They do not independently reconstruct NFDB labels, station-selection history, past-fire features, precipitation sums or source publication availability. The documented station-candidate selection issue and event-dependent edge-cell inclusion remain sensitivity questions; EDA does not resolve them.

In [ ]:
display(pd.DataFrame(summary["validation_violation_counts"].items(), columns=["check","violations"]))